In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

RULE_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.rule_results"
RULE_SCORE_TABLE = f"{CATALOG}.{SCHEMA}.rule_transaction_scores"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

RULE_RESULTS_PATH = (
    f"{S3_DELTA_PATH}/rule_results"
)

RULE_SCORE_PATH = (
    f"{S3_DELTA_PATH}/rule_transaction_scores"
)

# Separate checkpoint for Rule Engine
RULE_CHECKPOINT_PATH = (
    f"{S3_BASE_PATH}/checkpoints/rule_engine"
)


# ============================================================
# 2. RULE PARAMETERS
# ============================================================

HIGH_VALUE_THRESHOLD = 1000.0

# Simulated event-time window
VELOCITY_TIME_WINDOW = 2

VELOCITY_THRESHOLD = 5

FAN_IN_THRESHOLD = 5
FAN_OUT_THRESHOLD = 5

RULE_VERSION = "v1.0"

print("Incremental Rule Engine configuration loaded.")


# COMMAND ----------

# ============================================================
# 3. READ ONLY NEW SILVER RECORDS
# ============================================================

new_transactions_stream = (
    spark.readStream
        .format("delta")
        .table(SILVER_TX_TABLE)
)

print(f"Silver source: {SILVER_TX_TABLE}")


# COMMAND ----------

# ============================================================
# 4. PROCESS EACH NEW BATCH
# ============================================================
#
# foreachBatch allows us to:
#
#   1. Receive only newly arrived Silver rows.
#   2. Apply normal Spark batch transformations.
#   3. Use historical Silver data only when contextual
#      information is required by a rule.
#
# ============================================================

def process_rule_batch(new_transactions_df, batch_id):

    # --------------------------------------------------------
    # Ignore empty micro-batches
    # --------------------------------------------------------

    if new_transactions_df.isEmpty():
        print(f"Batch {batch_id}: no new transactions.")
        return

    print(f"Processing Rule Engine batch: {batch_id}")

    # --------------------------------------------------------
    # Count new transactions (cache not supported on serverless)
    # --------------------------------------------------------

    new_count = new_transactions_df.count()

    print(f"New Silver transactions in batch: {new_count}")

    # ========================================================
    # 5. RULE R001 - HIGH VALUE TRANSACTION
    # ========================================================

    rule_high_value = (
        new_transactions_df
        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time"
        )
        .withColumn(
            "rule_id",
            F.lit("R001")
        )
        .withColumn(
            "rule_name",
            F.lit("HIGH_VALUE_TRANSACTION")
        )
        .withColumn(
            "rule_category",
            F.lit("AMOUNT")
        )
        .withColumn(
            "rule_triggered",
            F.col("tx_amount") > HIGH_VALUE_THRESHOLD
        )
        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(30)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Transaction amount "),
                    F.col("tx_amount").cast("string"),
                    F.lit(" exceeds threshold "),
                    F.lit(str(HIGH_VALUE_THRESHOLD))
                )
            )
            .otherwise(F.lit(None))
        )
    )


    # ========================================================
    # 6. GET HISTORICAL CONTEXT ONLY WHERE REQUIRED
    # ========================================================
    #
    # We need historical data for R002/R003/R004.
    #
    # We do NOT run rules on all historical transactions.
    #
    # We first determine what event times are relevant to this
    # incoming batch, then retrieve only those event times.
    #
    # ========================================================

    relevant_event_times = (
        new_transactions_df
        .select("event_time")
        .distinct()
        .withColumn(
            "min_event_time",
            F.col("event_time") - F.lit(VELOCITY_TIME_WINDOW)
        )
        .withColumn(
            "max_event_time",
            F.col("event_time")
        )
    )

    historical_context = (
        spark.table(SILVER_TX_TABLE)
        .join(
            relevant_event_times,
            on=(
                (F.col("event_time") >= F.col("min_event_time")) &
                (F.col("event_time") <= F.col("max_event_time"))
            ),
            how="inner"
        )
        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time"
        )
        .dropDuplicates(["tx_id"])
    )

    # Combine historical context + new records
    context_transactions = (
        historical_context
        .unionByName(
            new_transactions_df.select(
                "tx_id",
                "sender_account_id",
                "receiver_account_id",
                "tx_amount",
                "event_time"
            ),
            allowMissingColumns=True
        )
        .dropDuplicates(["tx_id"])
    )


    # ========================================================
    # 7. RULE R002 - TRANSACTION VELOCITY
    # ========================================================
    #
    # IMPORTANT:
    # Past-looking window only.
    # No future transactions are used.
    #
    # ========================================================

    velocity_window = (
        Window
        .partitionBy("sender_account_id")
        .orderBy("event_time")
        .rangeBetween(
            -VELOCITY_TIME_WINDOW,
            0
        )
    )

    velocity_df = (
        context_transactions
        .withColumn(
            "velocity_count",
            F.count("tx_id").over(velocity_window)
        )
    )

    # Keep only incoming transactions for rule output
    rule_velocity = (
        velocity_df
        .join(
            new_transactions_df.select("tx_id"),
            on="tx_id",
            how="inner"
        )
        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "velocity_count"
        )
        .withColumn(
            "rule_id",
            F.lit("R002")
        )
        .withColumn(
            "rule_name",
            F.lit("TRANSACTION_VELOCITY")
        )
        .withColumn(
            "rule_category",
            F.lit("VELOCITY")
        )
        .withColumn(
            "rule_triggered",
            F.col("velocity_count") >= VELOCITY_THRESHOLD
        )
        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(20)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Sender executed "),
                    F.col("velocity_count").cast("string"),
                    F.lit(" transactions within time window")
                )
            )
            .otherwise(F.lit(None))
        )
    )


    # ========================================================
    # 8. RULE R003 - FAN IN
    # ========================================================
    #
    # Calculate incoming network activity using the relevant
    # historical context plus the new transactions.
    #
    # ========================================================

    fan_in_df = (
        context_transactions
        .groupBy(
            "receiver_account_id",
            "event_time"
        )
        .agg(
            F.countDistinct(
                "sender_account_id"
            ).alias("unique_senders"),

            F.sum(
                "tx_amount"
            ).alias("total_inflow")
        )
    )

    rule_fan_in = (
        new_transactions_df
        .join(
            fan_in_df,
            on=[
                "receiver_account_id",
                "event_time"
            ],
            how="left"
        )
        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "unique_senders",
            "total_inflow"
        )
        .withColumn(
            "rule_id",
            F.lit("R003")
        )
        .withColumn(
            "rule_name",
            F.lit("FAN_IN")
        )
        .withColumn(
            "rule_category",
            F.lit("NETWORK")
        )
        .withColumn(
            "rule_triggered",
            F.col("unique_senders") >= FAN_IN_THRESHOLD
        )
        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(25)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Receiver received funds from "),
                    F.col("unique_senders").cast("string"),
                    F.lit(" distinct senders at event time ")
                )
            )
            .otherwise(F.lit(None))
        )
    )


    # ========================================================
    # 9. RULE R004 - FAN OUT
    # ========================================================

    fan_out_df = (
        context_transactions
        .groupBy(
            "sender_account_id",
            "event_time"
        )
        .agg(
            F.countDistinct(
                "receiver_account_id"
            ).alias("unique_receivers"),

            F.sum(
                "tx_amount"
            ).alias("total_outflow")
        )
    )

    rule_fan_out = (
        new_transactions_df
        .join(
            fan_out_df,
            on=[
                "sender_account_id",
                "event_time"
            ],
            how="left"
        )
        .select(
            "tx_id",
            "sender_account_id",
            "receiver_account_id",
            "tx_amount",
            "event_time",
            "unique_receivers",
            "total_outflow"
        )
        .withColumn(
            "rule_id",
            F.lit("R004")
        )
        .withColumn(
            "rule_name",
            F.lit("FAN_OUT")
        )
        .withColumn(
            "rule_category",
            F.lit("NETWORK")
        )
        .withColumn(
            "rule_triggered",
            F.col("unique_receivers") >= FAN_OUT_THRESHOLD
        )
        .withColumn(
            "rule_score",
            F.when(
                F.col("rule_triggered"),
                F.lit(25)
            ).otherwise(F.lit(0))
        )
        .withColumn(
            "rule_evidence",
            F.when(
                F.col("rule_triggered"),
                F.concat(
                    F.lit("Sender transferred funds to "),
                    F.col("unique_receivers").cast("string"),
                    F.lit(" distinct receivers at event time ")
                )
            )
            .otherwise(F.lit(None))
        )
    )


    # ========================================================
    # 10. COMBINE RULE RESULTS
    # ========================================================

    rule_results_df = (
        rule_high_value
        .unionByName(
            rule_velocity,
            allowMissingColumns=True
        )
        .unionByName(
            rule_fan_in,
            allowMissingColumns=True
        )
        .unionByName(
            rule_fan_out,
            allowMissingColumns=True
        )
    )


    # ========================================================
    # 11. ADD AUDIT METADATA
    # ========================================================

    rule_results_df = (
        rule_results_df
        .withColumn(
            "rule_execution_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "rule_version",
            F.lit(RULE_VERSION)
        )
        .withColumn(
            "batch_id",
            F.lit(str(batch_id))
        )
    )


    # ========================================================
    # 12. KEEP ONLY TRIGGERED RULES
    # ========================================================

    triggered_rule_results_df = (
        rule_results_df
        .filter(
            F.col("rule_triggered") == True
        )
    )


    # ========================================================
    # 13. APPEND RULE RESULTS
    # ========================================================
    #
    # DO NOT overwrite.
    #
    # Existing historical rule results remain in the table.
    # Only the newly triggered rules are appended.
    #
    # ========================================================

    (
        triggered_rule_results_df
        .write
        .format("delta")
        .mode("append")
        .option(
            "path",
            RULE_RESULTS_PATH
        )
        .saveAsTable(RULE_RESULTS_TABLE)
    )


    # ========================================================
    # 14. CREATE SCORES FOR NEW TRANSACTIONS
    # ========================================================

    transaction_rule_scores = (
        triggered_rule_results_df
        .groupBy("tx_id")
        .agg(
            F.sum("rule_score")
                .alias("rule_score"),

            F.collect_set("rule_id")
                .alias("triggered_rule_ids"),

            F.collect_set("rule_name")
                .alias("triggered_rules")
        )
        .withColumn(
            "rule_execution_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "rule_version",
            F.lit(RULE_VERSION)
        )
        .withColumn(
            "batch_id",
            F.lit(str(batch_id))
        )
    )


    # ========================================================
    # 15. APPEND TRANSACTION RULE SCORES
    # ========================================================

    (
        transaction_rule_scores
        .write
        .format("delta")
        .mode("append")
        .option(
            "path",
            RULE_SCORE_PATH
        )
        .saveAsTable(RULE_SCORE_TABLE)
    )


    # ========================================================
    # 16. CLEAN UP
    # ========================================================

    print(
        f"Rule Engine batch {batch_id} completed successfully."
    )


# COMMAND ----------

# ============================================================
# 17. START INCREMENTAL RULE ENGINE
# ============================================================

query = (
    new_transactions_stream
        .writeStream
        .foreachBatch(process_rule_batch)
        .option(
            "checkpointLocation",
            RULE_CHECKPOINT_PATH
        )
        .trigger(
            availableNow=True
        )
        .start()
)


# ============================================================
# 18. WAIT FOR COMPLETION
# ============================================================

query.awaitTermination()

print("Incremental Rule Engine completed.")
print(f"Rule Results Table : {RULE_RESULTS_TABLE}")
print(f"Rule Score Table   : {RULE_SCORE_TABLE}")
print(f"Checkpoint         : {RULE_CHECKPOINT_PATH}")